In [1]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

# If running the notebook from data_labeling/eda/,
# this resolves the repository root correctly.
ROOT = Path.cwd().resolve()

while not (ROOT / "data_labeling").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Could not find the repository root.")
    ROOT = ROOT.parent

FULL_LABELS_CSV = (
    ROOT
    / "data_labeling"
    / "data_labels"
    / "noaa"
    / "labels_noaa_2010_2026_binary.csv"
)

STAGE_LABEL_DIR = (
    ROOT
    / "data_labeling"
    / "data_labels"
    / "noaa"
    / "CL_Labels"
)

IMAGE_DIR = ROOT / "downloaded_data" / "hmi_jpgs"

STAGE1_TRAIN_CSV = STAGE_LABEL_DIR / "Stage1_train.csv"
STAGE1_HOLDOUT_CSV = STAGE_LABEL_DIR / "Stage1_holdout.csv"

STAGE2_TRAIN_CSV = STAGE_LABEL_DIR / "Stage2_train.csv"
STAGE2_HOLDOUT_CSV = STAGE_LABEL_DIR / "Stage2_holdout.csv"

print("Repository root:", ROOT)
print("Full labels:", FULL_LABELS_CSV)
print("Stage label directory:", STAGE_LABEL_DIR)
print("Image directory:", IMAGE_DIR)

Repository root: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-
Full labels: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/noaa/labels_noaa_2010_2026_binary.csv
Stage label directory: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/noaa/CL_Labels
Image directory: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/downloaded_data/hmi_jpgs


In [2]:
# These boundaries are inclusive.

STAGE2_TRAIN_START = pd.Timestamp("2018-01-02 00:00:00")
STAGE2_TRAIN_END = pd.Timestamp("2023-03-08 22:00:00")

# This 24-hour interval is intentionally excluded.
STAGE2_GAP_START = pd.Timestamp("2023-03-08 23:00:00")
STAGE2_GAP_END = pd.Timestamp("2023-03-09 22:00:00")

STAGE2_HOLDOUT_START = pd.Timestamp("2023-03-09 23:00:00")
STAGE2_HOLDOUT_END = pd.Timestamp("2023-06-29 22:00:00")

assert STAGE2_TRAIN_END < STAGE2_GAP_START
assert STAGE2_GAP_END < STAGE2_HOLDOUT_START

print("Stage 2 train:", STAGE2_TRAIN_START, "to", STAGE2_TRAIN_END)
print("Excluded gap:  ", STAGE2_GAP_START, "to", STAGE2_GAP_END)
print("Stage 2 holdout:", STAGE2_HOLDOUT_START, "to", STAGE2_HOLDOUT_END)

Stage 2 train: 2018-01-02 00:00:00 to 2023-03-08 22:00:00
Excluded gap:   2023-03-08 23:00:00 to 2023-03-09 22:00:00
Stage 2 holdout: 2023-03-09 23:00:00 to 2023-06-29 22:00:00


In [3]:
data = pd.read_csv(FULL_LABELS_CSV)

required_columns = {"label", "goes_class"}
missing_columns = required_columns - set(data.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

timestamp_text = data["label"].str.extract(
    r"HMI\.m(\d{4}\.\d{2}\.\d{2}_\d{2}\.\d{2}\.\d{2})"
)[0]

if timestamp_text.isna().any():
    bad_rows = data.loc[timestamp_text.isna(), "label"].head(10).tolist()
    raise ValueError(f"Could not parse timestamps from labels: {bad_rows}")

data["timestamp"] = pd.to_datetime(
    timestamp_text,
    format="%Y.%m.%d_%H.%M.%S",
)

data["goes_class"] = data["goes_class"].astype(int)

# One row per downloaded magnetogram.
data = (
    data
    .drop_duplicates(subset="label", keep="last")
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print(f"Total unique labeled images: {len(data):,}")
print("Earliest timestamp:", data["timestamp"].min())
print("Latest timestamp:  ", data["timestamp"].max())

display(data.head())

Total unique labeled images: 128,117
Earliest timestamp: 2010-12-06 07:00:00
Latest timestamp:   2026-07-31 23:00:00


,label,goes_class,timestamp
0,2010/12/06/HMI.m2010.12.06_07.00.00.jpg,0,2010-12-06 07:00:00
1,2010/12/06/HMI.m2010.12.06_08.00.00.jpg,0,2010-12-06 08:00:00
2,2010/12/06/HMI.m2010.12.06_09.00.00.jpg,0,2010-12-06 09:00:00
3,2010/12/06/HMI.m2010.12.06_10.00.00.jpg,0,2010-12-06 10:00:00
4,2010/12/06/HMI.m2010.12.06_11.00.00.jpg,0,2010-12-06 11:00:00


In [4]:
stage2_train = data.loc[
    data["timestamp"].between(STAGE2_TRAIN_START, STAGE2_TRAIN_END)
].copy()

stage2_gap = data.loc[
    data["timestamp"].between(STAGE2_GAP_START, STAGE2_GAP_END)
].copy()

stage2_holdout = data.loc[
    data["timestamp"].between(STAGE2_HOLDOUT_START, STAGE2_HOLDOUT_END)
].copy()

assert not stage2_train.empty, "Stage 2 train is empty."
assert not stage2_holdout.empty, "Stage 2 holdout is empty."

print("Stage 2 train rows:", len(stage2_train))
print("Stage 2 gap rows:", len(stage2_gap))
print("Stage 2 holdout rows:", len(stage2_holdout))

Stage 2 train rows: 44125
Stage 2 gap rows: 24
Stage 2 holdout rows: 2651


In [5]:
def summarize(name, frame):
    counts = frame["goes_class"].value_counts().reindex([0, 1], fill_value=0)

    return {
        "split": name,
        "start": frame["timestamp"].min(),
        "end": frame["timestamp"].max(),
        "total": len(frame),
        "NF (0)": counts[0],
        "FL (1)": counts[1],
        "FL percent": round(100 * counts[1] / len(frame), 3),
    }

summary = pd.DataFrame(
    [
        summarize("Stage 2 train", stage2_train),
        summarize("Excluded 24h gap", stage2_gap),
        summarize("Stage 2 holdout", stage2_holdout),
    ]
)

display(summary)

,split,start,end,total,NF (0),FL (1),FL percent
0,Stage 2 train,2018-01-02 00:00:00,2023-03-08 22:00:00,44125,40344,3781,8.569
1,Excluded 24h gap,2023-03-08 23:00:00,2023-03-09 22:00:00,24,24,0,0.000
2,Stage 2 holdout,2023-03-09 23:00:00,2023-06-29 22:00:00,2651,1474,1177,44.398


In [6]:
stage1_train = pd.read_csv(STAGE1_TRAIN_CSV)
stage1_holdout = pd.read_csv(STAGE1_HOLDOUT_CSV)

stage1_labels = set(stage1_train["label"]) | set(stage1_holdout["label"])
stage2_labels = set(stage2_train["label"]) | set(stage2_holdout["label"])

stage1_stage2_overlap = stage1_labels & stage2_labels

assert not stage1_stage2_overlap, (
    f"Found {len(stage1_stage2_overlap)} overlap(s) between Stage 1 and Stage 2."
)

print("Passed: no image overlap between Stage 1 and Stage 2.")
print(f"Stage 1 images checked: {len(stage1_labels):,}")
print(f"Stage 2 images checked: {len(stage2_labels):,}")

Passed: no image overlap between Stage 1 and Stage 2.
Stage 1 images checked: 54,868
Stage 2 images checked: 46,776


In [8]:
# Images are stored only on the training server.
# Therefore, image-existence verification is intentionally deferred to the server.

stage2_paths = pd.concat(
    [stage2_train["label"], stage2_holdout["label"]],
    ignore_index=True,
)

print("Local image verification: skipped.")
print(f"Stage 2 image paths recorded: {len(stage2_paths):,}")
print("Verify these paths on the server before training Stage 2.")

Local image verification: skipped.
Stage 2 image paths recorded: 46,776
Verify these paths on the server before training Stage 2.


In [9]:
stage2_train_output = stage2_train[["label", "goes_class"]].copy()
stage2_holdout_output = stage2_holdout[["label", "goes_class"]].copy()

assert list(stage2_train_output.columns) == ["label", "goes_class"]
assert list(stage2_holdout_output.columns) == ["label", "goes_class"]

print("Stage 2 train preview:")
display(stage2_train_output.head())

print("Stage 2 holdout preview:")
display(stage2_holdout_output.head())

Stage 2 train preview:


,label,goes_class
54916,2018/01/02/HMI.m2018.01.02_00.00.00.jpg,0
54917,2018/01/02/HMI.m2018.01.02_01.00.00.jpg,0
54918,2018/01/02/HMI.m2018.01.02_02.00.00.jpg,0
54919,2018/01/02/HMI.m2018.01.02_03.00.00.jpg,0
54920,2018/01/02/HMI.m2018.01.02_04.00.00.jpg,0


Stage 2 holdout preview:


,label,goes_class
99065,2023/03/09/HMI.m2023.03.09_23.00.00.jpg,0
99066,2023/03/10/HMI.m2023.03.10_00.00.00.jpg,0
99067,2023/03/10/HMI.m2023.03.10_01.00.00.jpg,0
99068,2023/03/10/HMI.m2023.03.10_02.00.00.jpg,0
99069,2023/03/10/HMI.m2023.03.10_03.00.00.jpg,0


In [10]:
STAGE_LABEL_DIR.mkdir(parents=True, exist_ok=True)

if STAGE2_TRAIN_CSV.exists() or STAGE2_HOLDOUT_CSV.exists():
    raise FileExistsError(
        "Stage 2 file(s) already exist. Inspect them first rather than overwriting."
    )

stage2_train_output.to_csv(STAGE2_TRAIN_CSV, index=False)
stage2_holdout_output.to_csv(STAGE2_HOLDOUT_CSV, index=False)

print("Created:", STAGE2_TRAIN_CSV)
print("Created:", STAGE2_HOLDOUT_CSV)

Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/noaa/CL_Labels/Stage2_train.csv
Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/noaa/CL_Labels/Stage2_holdout.csv
